# Ensemble

Autor: Bartosz Perz

## Zadanie 1: Salud 2030

Na potrzeby zadania potraktuj dane jako w pełni zebrane od ankietowanych.

### Wprowadzenie

Pracujesz jako Senior Data Scientist w **Secretaría de Salud de México** – meksykańskim Ministerstwie Zdrowia. Rząd Meksyku przeznaczył ogromne fundusze, angażując Cię do pomocy w wywiązaniu się z globalnych zobowiązań wobec ONZ i WHO.

Meksyk, jako sygnatariusz Agendy 2030, kładzie szczególny nacisk na **3. Cel Zrównoważonego Rozwoju (SDG 3): Dobre zdrowie i jakość życia**. Kluczowym elementem tego celu jest **Zadanie 3.4**, które zakłada ograniczenie do 2030 roku o jedną trzecią przedwczesnej umieralności z powodu chorób niezakaźnych (NCDs) poprzez zapobieganie i leczenie.

Jak jednak spełnić to wymaganie? Bierzesz na celownik otyłość – główny czynnik ryzyka wystąpienia chorób układu krążenia, cukrzycy typu 2 i nadciśnienia, które stanowią trzon problemów zdrowotnych wymienionych w SDG 3. Twoim celem jest stworzenie modelu, który na podstawie danych ankietowych zidentyfikuje osoby wymagające interwencji profilaktycznej. Aby pozyskać zbiór danych do uczenia modeli, w wybranych przychodniach przeprowadzono badanie pilotażowe.

Musisz działać precyzyjnie, ponieważ niedokładny model pominie zagrożonych pacjentów, a model zbyt "ciężki" obliczeniowo nie będzie mógł zostać wdrożony w wiejskich przychodniach o ograniczonych zasobach. W Meksyku, gdzie wskaźniki otyłości należą do najwyższych na świecie, Twoja praca nad optymalnym modelem predykcyjnym jest bezpośrednim wkładem w walkę o wydłużenie życia obywateli i redukcję obciążenia systemu ochrony zdrowia.

### Polecenia:

1. **Wykonaj analizę eksploracyjną (EDA)** [zbioru danych dot. otyłości (UCI ID: 544)](https://archive.ics.uci.edu/dataset/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition).
2. **Wyucz modele klasyfikacyjne**. Wykorzystaj bibliotekę `scikit-learn` i zbadaj:

* **Pojedyncze drzewa decyzyjne** i **regresję logistyczną** – jako bazowe modele, łatwe do interpretacji przez lekarza.
* **Bagging:** `BaggingClassifier` z płytkimi drzewami, głębokimi drzewami oraz z regresją logistyczną. Przetestuj wpływ parametrów *n_estimators*, *bootstrap* i *bootstrap_features* na wyniki.
* **Boosting:** `AdaBoost` (przetestuj na płytkich drzewach i regresji logistycznej) oraz `GradientBoosting` (oparty domyślnie na drzewach). Zbadaj relację (trade-off) między parametrami *n_estimators* i *learning_rate*.
* **Stacking & Voting:** skonstruuj kolejny komitet wg własnego uznania (dobierz parametry i modele). Połącz modele o różnej architekturze, by sprawdzić, czy zespół różnych algorytmów daje lepsze efekty niż pojedynczy "ekspert".

3. **Zwaliduj modele** pod kątem wdrożenia krajowego.

* Dobierz miary **skuteczności** modelu adekwatne do problemu.
* Zwróć uwagę na **czas predykcji**.
* Zbadaj **pewność (confidence) modelu**.

4. **Zapisz wnioski dla Ministerstwa Zdrowia.**

Przy realizacji poleceń pamiętaj o charakterze zadania. Który z modeli zarekomendowałbyś do wdrożenia? Czy dokładność modelu jest jedynym wyznacznikiem jego jakości? Czy są modele, których wdrożenie byłoby stratą zasobów publicznych?

### 0. importy

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OrdinalEncoder, OneHotEncoder, StandardScaler, LabelEncoder
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    BaggingClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier,
    RandomForestClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")
np.random.seed(42)

### 1. EDA - analiza

In [ ]:
# wczytanie danych
df = pd.read_csv("./data/zad1_obesity.csv")

print("Kształt zbioru:", df.shape)
print("\nRozkład klas:")
print(df["NObeyesdad"].value_counts())

In [ ]:
# przeglad ogolny
print("=== Informacje o zbiorze ===")
df.info()
print("\n=== Statystyki opisowe (numeryczne) ===")
df.describe().round(2)

In [ ]:
# rozklad zmiennej docelowej
fig, ax = plt.subplots(figsize=(9, 4))
order = df["NObeyesdad"].value_counts().index

sns.countplot(data=df, y="NObeyesdad", order=order, palette="viridis", ax=ax)
ax.set_title("Rozkład klas otyłości (zmienna docelowa)", fontsize=13)
ax.set_xlabel("Liczba obserwacji")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# rozklad cech numerycznych
num_cols = ["Age", "Height", "Weight", "FCVC", "NCP", "CH2O", "FAF", "TUE"]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flatten(), num_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(col)
    ax.set_xlabel("")
plt.suptitle("Rozkłady cech numerycznych", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# cechy kategoryczne
cat_cols = [
    "Gender", "family_history_with_overweight", "FAVC",
    "CAEC", "SMOKE", "SCC", "CALC", "MTRANS"
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax, palette="Set2")
    ax.set_title(col)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
    ax.set_xlabel("")
plt.suptitle("Rozkłady cech kategorycznych", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# macierz korelacji
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Macierz korelacji – cechy numeryczne", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# waga i wzrost wg klasy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = [
    "Insufficient_Weight", "Normal_Weight",
    "Overweight_Level_I", "Overweight_Level_II",
    "Obesity_Type_I", "Obesity_Type_II", "Obesity_Type_III"
]

sns.boxplot(data=df, x="NObeyesdad", y="Weight", order=order,
            palette="RdYlGn_r", ax=axes[0])
axes[0].set_title("Waga wg klasy otyłości")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=35, ha="right")

sns.boxplot(data=df, x="NObeyesdad", y="Height", order=order,
            palette="RdYlGn_r", ax=axes[1])
axes[1].set_title("Wzrost wg klasy otyłości")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=35, ha="right")

plt.suptitle("Rozkład wagi i wzrostu wg kategorii BMI", fontsize=13)
plt.tight_layout()
plt.show()

### 2. Preprocessing i podzial danych

In [ ]:
# Kolumny binarne (yes/no)
binary_cols = ["family_history_with_overweight", "FAVC", "SMOKE", "SCC"]

# Kolumny
caec_order  = ["no", "Sometimes", "Frequently", "Always"]
calc_order  = ["no", "Sometimes", "Frequently", "Always"]

# Kolumny nominalne (OHE)
nominal_cols = ["Gender", "MTRANS"]

# Kolumny numeryczne (bez zmian dla drzew; scaler dla LR)
num_cols = ["Age", "Height", "Weight", "FCVC", "NCP", "CH2O", "FAF", "TUE"]

# Preprocessor BEZ skalowania (dla drzew/ensemble)
preprocessor_tree = ColumnTransformer(transformers=[
    ("num",    "passthrough",                                              num_cols),
    ("bin",    OrdinalEncoder(),                                           binary_cols),
    ("caec",   OrdinalEncoder(categories=[caec_order]),                   ["CAEC"]),
    ("calc",   OrdinalEncoder(categories=[calc_order]),                   ["CALC"]),
    ("nom",    OneHotEncoder(drop="first", sparse_output=False),          nominal_cols),
], remainder="drop")

# Preprocessor ZE skalowaniem (dla regresji logistycznej)
preprocessor_lr = ColumnTransformer(transformers=[
    ("num",    StandardScaler(),                                           num_cols),
    ("bin",    OrdinalEncoder(),                                           binary_cols),
    ("caec",   OrdinalEncoder(categories=[caec_order]),                   ["CAEC"]),
    ("calc",   OrdinalEncoder(categories=[calc_order]),                   ["CALC"]),
    ("nom",    OneHotEncoder(drop="first", sparse_output=False),          nominal_cols),
], remainder="drop")

# Podział zbioru
X = df.drop(columns=["NObeyesdad"])
y = df["NObeyesdad"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

print(f"Zbiór treningowy: {X_train.shape[0]} próbek")
print(f"Zbiór testowy:    {X_test.shape[0]} próbek")

In [ ]:
results = {}  # słownik do przechowywania wyników wszystkich modeli

def evaluate_model(name, pipeline, X_train, X_test, y_train, y_test, cv=5):
    """Trenuje pipeline, mierzy czas predykcji, accuracy, F1 macro i confidence."""
    pipeline.fit(X_train, y_train)

    # Czas predykcji (średnia z 50 powtórzeń)
    times = []
    for _ in range(50):
        t0 = time.perf_counter()
        pipeline.predict(X_test)
        times.append(time.perf_counter() - t0)
    inference_ms = np.mean(times) * 1000  # ms

    y_pred = pipeline.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    f1     = f1_score(y_test, y_pred, average="macro")

    # Confidence (tylko jeśli model obsługuje predict_proba)
    if hasattr(pipeline, "predict_proba"):
        proba      = pipeline.predict_proba(X_test)
        confidence = float(np.mean(np.max(proba, axis=1)))
    else:
        confidence = float("nan")

    # Cross-validation accuracy
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="accuracy")

    results[name] = {
        "Accuracy":       round(acc, 4),
        "F1 macro":       round(f1, 4),
        "CV Acc (mean)":  round(cv_scores.mean(), 4),
        "CV Acc (std)":   round(cv_scores.std(), 4),
        "Confidence":     round(confidence, 4),
        "Inference (ms)": round(inference_ms, 3),
    }

    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Accuracy       : {acc*100:.2f}%")
    print(f"  F1 macro       : {f1*100:.2f}%")
    print(f"  CV Acc         : {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")
    print(f"  Confidence     : {confidence*100:.2f}%")
    print(f"  Inference time : {inference_ms:.3f} ms")

    return pipeline

### 3. Modele bazowe

In [ ]:
# 1. Drzewo decyzyjne (bazowe)
dt_pipe = Pipeline([
    ("prep", preprocessor_tree),
    ("clf",  DecisionTreeClassifier(random_state=42))
])
dt_pipe = evaluate_model("Drzewo decyzyjne (baseline)", dt_pipe,
                          X_train, X_test, y_train, y_test)

# 2. Regresja logistyczna (bazowa)
lr_pipe = Pipeline([
    ("prep", preprocessor_lr),
    ("clf",  LogisticRegression(max_iter=1000, random_state=42))
])
lr_pipe = evaluate_model("Regresja logistyczna (baseline)", lr_pipe,
                          X_train, X_test, y_train, y_test)

### 4. Bagging

In [ ]:
# 3. Bagging: płytkie drzewo
bag_shallow_pipe = Pipeline([
    ("prep", preprocessor_tree),
    ("clf",  BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=3, random_state=42),
        n_estimators=50, bootstrap=True, bootstrap_features=False, random_state=42
    ))
])
bag_shallow_pipe = evaluate_model("Bagging – płytkie drzewo (depth=3, n=50)",
                                   bag_shallow_pipe, X_train, X_test, y_train, y_test)

# 4. Bagging: głębokie drzewo
bag_deep_pipe = Pipeline([
    ("prep", preprocessor_tree),
    ("clf",  BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=None, random_state=42),
        n_estimators=50, bootstrap=True, bootstrap_features=False, random_state=42
    ))
])
bag_deep_pipe = evaluate_model("Bagging – głębokie drzewo (depth=None, n=50)",
                                bag_deep_pipe, X_train, X_test, y_train, y_test)

# 5. Bagging: regresja logistyczna
bag_lr_pipe = Pipeline([
    ("prep", preprocessor_lr),
    ("clf",  BaggingClassifier(
        estimator=LogisticRegression(max_iter=1000, random_state=42),
        n_estimators=20, bootstrap=True, bootstrap_features=True, random_state=42
    ))
])
bag_lr_pipe = evaluate_model("Bagging – regresja logistyczna (n=20, boot_feat=True)",
                              bag_lr_pipe, X_train, X_test, y_train, y_test)

In [ ]:
# Wpływ parametrów Baggingu (wykres)
n_range     = [5, 10, 20, 50, 100]
acc_boot    = []
acc_noboot  = []

prep = preprocessor_tree
prep.fit(X_train)
X_tr_t = prep.transform(X_train)
X_te_t = prep.transform(X_test)

for n in n_range:
    for boot, store in [(True, acc_boot), (False, acc_noboot)]:
        m = BaggingClassifier(
            estimator=DecisionTreeClassifier(max_depth=3, random_state=42),
            n_estimators=n, bootstrap=boot, random_state=42
        )
        m.fit(X_tr_t, y_train)
        store.append(accuracy_score(y_test, m.predict(X_te_t)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_range, acc_boot,   "o-", label="bootstrap=True")
ax.plot(n_range, acc_noboot, "s--", label="bootstrap=False")
ax.set_xlabel("n_estimators")
ax.set_ylabel("Accuracy (test)")
ax.set_title("Bagging: wpływ n_estimators i bootstrap (płytkie drzewo)")
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

### 5. Boosting

In [ ]:
# 6. AdaBoost: płytkie drzewo
ada_dt_pipe = Pipeline([
    ("prep", preprocessor_tree),
    ("clf",  AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=3, random_state=42),
        n_estimators=100, learning_rate=0.5, random_state=42
    ))
])
ada_dt_pipe = evaluate_model("AdaBoost – płytkie drzewo (n=100, lr=0.5)",
                              ada_dt_pipe, X_train, X_test, y_train, y_test)

# 7. AdaBoost: regresja logistyczna
ada_lr_pipe = Pipeline([
    ("prep", preprocessor_lr),
    ("clf",  AdaBoostClassifier(
        estimator=LogisticRegression(max_iter=1000, random_state=42),
        n_estimators=50, learning_rate=1.0, random_state=42
    ))
])
ada_lr_pipe = evaluate_model("AdaBoost – regresja logistyczna (n=50, lr=1.0)",
                              ada_lr_pipe, X_train, X_test, y_train, y_test)

In [ ]:
# GradientBoosting: trade-off n_estimators vs learning_rate

# Siatka parametrów
lr_values = [0.01, 0.05, 0.1, 0.2, 0.5]
n_values  = [50, 100, 200]

prep_gb = preprocessor_tree
prep_gb.fit(X_train)
X_tr_gb = prep_gb.transform(X_train)
X_te_gb = prep_gb.transform(X_test)

gb_results = []
for n in n_values:
    for lr_val in lr_values:
        m = GradientBoostingClassifier(
            n_estimators=n, learning_rate=lr_val,
            max_depth=3, random_state=42
        )
        m.fit(X_tr_gb, y_train)
        acc = accuracy_score(y_test, m.predict(X_te_gb))
        gb_results.append({"n_estimators": n, "learning_rate": lr_val, "accuracy": acc})

gb_df = pd.DataFrame(gb_results)
pivot  = gb_df.pivot(index="learning_rate", columns="n_estimators", values="accuracy")

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGn", ax=ax)
ax.set_title("GradientBoosting: Accuracy vs learning_rate i n_estimators")
ax.set_xlabel("n_estimators")
ax.set_ylabel("learning_rate")
plt.tight_layout()
plt.show()

# Najlepsza konfiguracja
best_row = gb_df.loc[gb_df["accuracy"].idxmax()]
print(f"\nNajlepsza konfiguracja GB: n={int(best_row.n_estimators)}, "
      f"lr={best_row.learning_rate}, acc={best_row.accuracy:.4f}")

In [ ]:
# GradientBoosting (najlepsza konfiguracja)
gb_pipe = Pipeline([
    ("prep", preprocessor_tree),
    ("clf",  GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1,
        max_depth=3, random_state=42
    ))
])
gb_pipe = evaluate_model("GradientBoosting (n=200, lr=0.1, depth=3)",
                          gb_pipe, X_train, X_test, y_train, y_test)

### 6. Stacking i Voting

In [ ]:
# Voting classifier

voting_pipe = Pipeline([
    ("prep", preprocessor_tree),
    ("clf",  VotingClassifier(
        estimators=[
            ("dt",  DecisionTreeClassifier(max_depth=8, random_state=42)),
            ("rf",  RandomForestClassifier(n_estimators=100, random_state=42)),
            ("lr",  LogisticRegression(max_iter=5000, random_state=42)),
        ],
        voting="soft"
    ))
])
voting_pipe = evaluate_model("VotingClassifier – soft (DT + RF + LR)",
                              voting_pipe, X_train, X_test, y_train, y_test)

In [ ]:
# Stacking Classifier
stacking_pipe = Pipeline([
    ("prep", preprocessor_tree),
    ("clf",  StackingClassifier(
        estimators=[
            ("dt",   DecisionTreeClassifier(max_depth=6, random_state=42)),
            ("bag",  BaggingClassifier(
                         estimator=DecisionTreeClassifier(max_depth=5, random_state=42),
                         n_estimators=30, random_state=42)),
            ("ada",  AdaBoostClassifier(
                         estimator=DecisionTreeClassifier(max_depth=3, random_state=42),
                         n_estimators=50, random_state=42)),
        ],
        final_estimator=LogisticRegression(max_iter=2000, random_state=42),
        cv=5,
        passthrough=False,
    ))
])
stacking_pipe = evaluate_model("StackingClassifier (DT + Bagging + AdaBoost → LR)",
                                stacking_pipe, X_train, X_test, y_train, y_test)

### 7. Walidacja i porównanie modeli

In [ ]:
# tabela wynikow
results_df = pd.DataFrame(results).T.sort_values("Accuracy", ascending=False)
results_df.index.name = "Model"
print("\n=== PORÓWNANIE WSZYSTKICH MODELI ===\n")
print(results_df.to_string())

In [ ]:
# Wykres: Accuracy vs Inference time
fig, ax = plt.subplots(figsize=(10, 6))

colors = plt.cm.tab10(np.linspace(0, 1, len(results_df)))

for (model_name, row), color in zip(results_df.iterrows(), colors):
    ax.scatter(row["Inference (ms)"], row["Accuracy"],
               s=120, color=color, zorder=5)
    ax.annotate(model_name,
                xy=(row["Inference (ms)"], row["Accuracy"]),
                xytext=(6, 3), textcoords="offset points",
                fontsize=7.5, color=color)

ax.set_xlabel("Czas predykcji (ms)", fontsize=11)
ax.set_ylabel("Accuracy (test)", fontsize=11)
ax.set_title("Trade-off: dokładność vs czas predykcji", fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix dla najlepszego modelu
# Zakładamy GradientBoosting jako kandydat do wdrożenia
best_pipe = gb_pipe

fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_estimator(
    best_pipe, X_test, y_test,
    display_labels=best_pipe.classes_,
    cmap="Blues",
    xticks_rotation=35,
    ax=ax
)
ax.set_title("Confusion Matrix – GradientBoosting (n=200, lr=0.1)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Classification report najlepszego modelu
y_pred_best = best_pipe.predict(X_test)
print("=== Classification Report – GradientBoosting ===\n")
print(classification_report(y_test, y_pred_best))

### 8. Wnioski

In [ ]:
print("""
WNIOSKI do zadania 1
==========================================

1. REKOMENDACJA DO WDROŻENIA
   GradientBoosting (n=200, lr=0.1, max_depth=3)

   - Najwyższa accuracy (95.74%) i F1 macro (95.70%) spośród
     wszystkich testowanych modeli
   - Najwyższa pewność predykcji (97.01%) – lekarz może ocenić,
     kiedy model jest niepewny i skierować pacjenta na dalsze
     badania zamiast polegać wyłącznie na wyniku algorytmu
   - Stabilny (CV: 96.21% ± 0.96%) – niska wariancja oznacza,
     że model generalizuje dobrze na nowych pacjentach
   - Czas predykcji (9.1 ms) akceptowalny – model trenowany
     offline centralnie, eksportowany do przychodni jako plik


2. MODELE DO ODRZUCENIA / TYLKO REFERENCYJNE

   - AdaBoost z regresją logistyczną (49.21%): kompletna
     porażka – gorszy od losowego klasyfikatora dla 7 klas.
     LR jako model bazowy dla AdaBoost jest zbyt słaby dla
     tego nieliniowego problemu
   - Bagging z płytkimi drzewami (69.56%): zaskakująco słaby.
     Płytkie drzewa (max_depth=3) jako baza Baggingu są zbyt
     niedopasowane (high bias) – komitet wzmacnia błąd, nie
     wariancję. Bagging z głębokimi drzewami (94.16%) działa
     znacznie lepiej, co potwierdza tę diagnozę
   - StackingClassifier (88.33%): gorszy od drzewa bazowego
     (90.06%). Modele bazowe (DT, Bagging, AdaBoost) nie są
     wystarczająco zróżnicowane, by meta-klasyfikator mógł
     wyciągnąć dodatkową wartość
   - Drzewo decyzyjne i regresja logistyczna: użyteczne jako
     modele referencyjne i do interpretacji przez lekarza,
     ale dokładność (90.06% / 86.59%) niewystarczająca do
     wdrożenia krajowego


3. UWAGI PRAKTYCZNE

   - Accuracy nie jest jedynym wyznacznikiem – w kontekście
     medycznym kluczowy jest recall dla klas otyłości: lepiej
     zidentyfikować kogoś jako zagrożonego, niż przeoczyć przypadek
   - F1 macro zbliżone do Accuracy dla GB oznacza, że model radzi
     sobie równomiernie ze wszystkimi 7 klasami – brak faworyzowania
     klas najliczniejszych
   - AdaBoost z drzewem (91.48%) ma najdłuższy czas predykcji
     (12.1 ms) przy gorszym wyniku niż GB – niekorzystny trade-off
""")

## Zadanie 2: Grzybobranie

### Wprowadzenie

Mieszkasz w "Warszawce". Ostatnio całe miasto opanował nowy trend – grzybobranie! Czy wiedziałeś, że grzybów nie trzeba kupować w sklepie? Rosną w lesie, zupełnie za darmo i E-KO-LO-GICZ-NIE.

Postanawiasz wybrać się do "mało znanego" miejsca – lasu pod Konstancinem-Jeziorną. Niestety, na miejscu okazuje się, że o grzybach nie masz zielonego pojęcia, a odróżnienie borowika od muchomora stanowi dla Ciebie ogromne wyzwanie. Już masz się poddać, ale nagle słyszysz za sobą podniesiony głos:

– Radziu, nie mów do mnie teraz!

Obracasz się i widzisz miło wyglądającą rodzinkę.

<img src="img.jpg" width="400" height="400" />

(Źródło: <a href="https://www.instagram.com/p/COLCA3RHoni/">instagram m_rozenek</a>)

Skądś ich kojarzysz (może z *morning matcha rave*?), ale nie to jest teraz najważniejsze. Widzisz, że próbują rozszyfrować **ZAAWANSOWANY ATLAS GRZYBÓW**. Postanawiasz dołączyć do zadania, ale szybko okazuje się, że rozumiesz z niego tyle co nic. Zamiast zdjęć i opisów, atlas zawiera wyłącznie tabele przedstawiające cechy grzybów. Sprawdzanie całej tabeli w poszukiwaniu odpowiedniego dopasowania do zaobserwowanych parametrów okazu od razu wydaje Ci się marnowaniem czasu. Wykorzystujesz więc wiedzę nabytą na zajęciach ze Sztucznej Inteligencji i postanawiasz stworzyć model uczenia maszynowego klasyfikujący grzyby.

Pamiętaj – błąd algorytmu oznacza poważne zatrucie. Chociaż masz dostęp do potężnych narzędzi, bateria w Twoim telefonie jest na wyczerpaniu, a każda sekunda pracy procesora jest na wagę złota. Musisz zdecydować, jaki model będzie adekwatny do zadania – na tyle precyzyjny, by przeżyć, a zarazem wystarczająco "lekki", by zadziałał w lesie.

### Polecenia:

1. **Wykonaj analizę eksploracyjną (EDA)** [zbioru danych o grzybach (UCI ID: 73)](https://archive.ics.uci.edu/dataset/73/mushroom).
2. Według własnego uznania **wybierz modele** i **wyucz** je. Wykorzystaj zdobytą dotychczas wiedzę. Przynajmniej jednym z testowanych modeli powinien być model zespołowy (ensemble).
3. **Zwaliduj wyuczone modele**. Pamiętaj, że pomyłka może skończyć się zatruciem, więc warto byłoby móc zinterpretować i zrozumieć decyzje modelu.
4. **Zapisz wnioski**.

Przy realizacji poleceń pamiętaj o specyficznym charakterze i ograniczeniach zadania.